# Cubic Membrane Photonics — Interactive Demo

This notebook walks through the full six-phase simulation pipeline for modelling curvature-driven cubic membrane templating and photonic crystal formation in beetle scales.

**Pipeline overview:**

| Phase | Description |
|-------|-------------|
| 1 | Baseline Allen-Cahn phase-field model (no proteins) |
| 2 | Spontaneous curvature term (protein loading P sweep) |
| 3 | Symmetry identification (gyroid vs diamond) |
| 4 | Lattice scaling to visible-light regime |
| 5 | Photonic band structure (plane-wave expansion) |
| 6 | Polycrystalline domain assembly |

---

**To run:** Execute cells in order (`Shift+Enter`). All figures are saved to `figures/` and data to `data/`.

In [ ]:
import sys, os
from pathlib import Path

# Add src/ to path
REPO = Path(os.getcwd())
sys.path.insert(0, str(REPO / 'src'))

FIG  = REPO / 'figures'
DATA = REPO / 'data'
FIG.mkdir(exist_ok=True)
DATA.mkdir(exist_ok=True)

import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless backend — change to 'inline' for interactive display
import matplotlib.pyplot as plt
from IPython.display import Image, display

print('Environment ready.')
print(f'  Figures -> {FIG}')
print(f'  Data    -> {DATA}')

---
## Phase 1 — Baseline Phase-Field Model

The Allen-Cahn equation evolves a scalar order parameter `phi(r,t)` on a 3-D periodic grid:

$$\frac{\partial \phi}{\partial t} = \lambda \nabla^2 \phi + \phi - \phi^3$$

A gyroid-inspired cosine superposition is used as the initial condition:

$$\phi_0(\mathbf{r}) = A_0 \left[ \cos(mx)\sin(my) + \cos(my)\sin(mz) + \cos(mz)\sin(mx) \right]$$

The equation is solved spectrally using a semi-implicit Fourier-space stepper:

$$\hat{\phi}^{n+1} = \frac{\hat{\phi}^n + \Delta t \cdot (-\phi^3)^n_{\mathbf{k}}}{1 - \Delta t (\lambda k^2 + 1)}$$

In [ ]:
from phase1_baseline import run_baseline, plot_isosurface_slice

print('Running Phase 1 (N=32, 200 steps)...')
phi_baseline = run_baseline(
    N=32, lam=0.1, dt=0.05, n_steps=200,
    seed=42, save_snapshots=True, out_dir=FIG
)
plot_isosurface_slice(phi_baseline, FIG, tag='baseline')
np.save(DATA / 'phi_baseline.npy', phi_baseline)

print(f'\nFinal field statistics:')
print(f'  min = {phi_baseline.min():.4f}')
print(f'  max = {phi_baseline.max():.4f}')
print(f'  std = {phi_baseline.std():.4f}')

In [ ]:
display(Image(str(FIG / 'phase1_baseline_snapshots.png')))

In [ ]:
display(Image(str(FIG / 'phase1_baseline_slices.png')))

---
## Phase 2 — Spontaneous Curvature (Protein Loading Sweep)

Protein loading `P in [0,1]` introduces a spontaneous curvature `H0(P) = alpha * P` that modifies the effective interface parameter:

$$\lambda_{\text{eff}}(P) = \lambda + \kappa(P) \cdot H_0(P), \quad \kappa(P) = \kappa_0(1 + \beta P)$$

As `P` increases, `lam_eff` grows, shifting the dominant wavenumber and driving morphology transitions from lamellar to bicontinuous cubic phases.

In [ ]:
from phase2_curvature import sweep_protein_loading, effective_lam, spontaneous_curvature

P_values = [0.0, 0.25, 0.5, 0.75, 1.0]

print('Protein loading sweep parameters:')
print(f'{"P":>6}  {"H0":>8}  {"lam_eff":>10}')
for P in P_values:
    print(f'{P:6.2f}  {spontaneous_curvature(P):8.4f}  {effective_lam(P):10.4f}')

print('\nRunning Phase 2...')
phi_sweep = sweep_protein_loading(
    P_values, N=32, lam=0.1, dt=0.05, n_steps=200, seed=42, out_dir=FIG
)

for P, phi in phi_sweep.items():
    np.save(DATA / f'phi_P{int(P*100):03d}.npy', phi)

print('Phase 2 complete.')

In [ ]:
display(Image(str(FIG / 'phase2_curvature_sweep.png')))

---
## Phase 3 — Symmetry Identification

Each morphology is classified using:
1. **Euler characteristic** `chi` of the thresholded isosurface (Gauss-Bonnet theorem on discrete surface)
2. **Spherically averaged power spectrum** `S(|k|)` — peak ratios distinguish gyroid (sqrt(6)) from diamond (sqrt(3))

| chi | Dominant peaks | Morphology |
|-----|---------------|------------|
| 0 | Single | Lamellar |
| -4 | sqrt(6) ratio | Gyroid (Ia3d) |
| -8 | sqrt(3) ratio | Double-diamond (Pn3m) |
| < -8 | Broad | Disordered bicontinuous |

In [ ]:
from phase3_symmetry import classify_morphology, plot_power_spectra, plot_phase_diagram

print('Running Phase 3 symmetry identification...')
print(f'{"P":>6}  {"chi":>8}  {"Morphology":>20}')

classifications = {}
for P, phi in phi_sweep.items():
    result = classify_morphology(phi)
    classifications[P] = result
    print(f'{P:6.2f}  {result["chi"]:8d}  {result["morphology"]:>20}')

plot_power_spectra(phi_sweep, classifications, out_dir=FIG)
plot_phase_diagram(classifications, out_dir=FIG)
print('Phase 3 complete.')

In [ ]:
display(Image(str(FIG / 'phase3_power_spectra.png')))

In [ ]:
display(Image(str(FIG / 'phase3_phase_diagram.png')))

---
## Phase 4 — Lattice Scaling to Visible-Light Regime

The simulation lattice constant `a_sim` is mapped to physical units via the Helfrich length scale:

$$\xi(P) = \sqrt{\kappa(P) / \sigma}, \quad a_{\text{nm}}(P) = a_{\text{sim}} \cdot \xi(P) \cdot C$$

The calibration constant `C` is set so that `P=1` gives `a = 350 nm` (visible red). The target range for beetle photonic crystals is 200-500 nm.

In [ ]:
from phase4_scaling import compute_lattice_scaling

print('Running Phase 4 lattice scaling...')
scaling = compute_lattice_scaling(phi_sweep, out_dir=FIG)

print(f'\n{"P":>6}  {"a_sim":>8}  {"xi":>8}  {"a_nm":>10}  {"Status":>12}')
for P, row in scaling.items():
    status = 'visible' if 200 <= row['a_nm'] <= 700 else 'out of range'
    print(f'{P:6.2f}  {row["a_sim"]:8.3f}  {row["xi"]:8.3f}  {row["a_nm"]:8.1f} nm  {status:>12}')

print('Phase 4 complete.')

In [ ]:
display(Image(str(FIG / 'phase4_lattice_scaling.png')))

---
## Phase 5 — Photonic Band Structure

The thresholded phase field is mapped to a dielectric function `epsilon(r)` (chitin: `eps=2.56`, air: `eps=1`). The photonic band structure is computed via the plane-wave expansion (PWE) method:

$$\sum_{\mathbf{G}'} M(\mathbf{G},\mathbf{G}') H(\mathbf{G}') = \left(\frac{\omega}{c}\right)^2 H(\mathbf{G})$$

along the path `Gamma -> X -> M -> Gamma -> R` of the simple cubic Brillouin zone.

> **Note:** `n_pw=2` (125 plane waves) is used here for speed. For accurate band gaps, use `n_pw=5` or higher.

In [ ]:
from phase5_photonics import run_photonics

print('Running Phase 5 photonic band structure...')
for P in [0.5, 1.0]:
    phi = phi_sweep[P]
    a_nm = scaling[P]['a_nm']
    morph = classifications[P]['morphology']
    print(f'  P={P:.1f} ({morph}), a={a_nm:.0f} nm ...')
    run_photonics(
        phi, P=P, a_nm=a_nm, morphology=morph,
        n_pw=2, n_kpoints=8, out_dir=FIG
    )

print('Phase 5 complete.')

In [ ]:
display(Image(str(FIG / 'phase5_band_structure_P100_diamond.png')))

---
## Phase 6 — Polycrystalline Domain Assembly

A Voronoi tessellation partitions the simulation box into `n_domains` independent grains, each with a randomly drawn protein loading `P_i`. Independent phase-field simulations are run per domain and assembled into a global polycrystalline field.

This models the mosaic of crystalline domains observed in beetle wing scales, where each domain is a single-crystal photonic grain of size 3-15 um.

In [ ]:
from phase6_polycrystal import run_polycrystal

print('Running Phase 6 polycrystalline assembly (6 domains)...')
phi_poly, domain_map, domain_info = run_polycrystal(
    N=32, n_domains=6, n_steps=200, dt=0.05, lam=0.1,
    P_min=0.4, P_max=1.0, seed=42, out_dir=FIG
)

print(f'\n{"Domain":>8}  {"P":>6}  {"Size (um)":>12}')
for d in domain_info:
    print(f'{d["id"]:8d}  {d["P"]:6.3f}  {d["size_um"]:10.2f} um')

np.save(DATA / 'phi_polycrystal.npy', phi_poly)
print('Phase 6 complete.')

In [ ]:
display(Image(str(FIG / 'phase6_polycrystal_slices.png')))

In [ ]:
display(Image(str(FIG / 'phase6_domain_analysis.png')))

---
## Full Pipeline Summary

In [ ]:
display(Image(str(FIG / 'summary_all_phases.png')))

---
## Next Steps

To improve accuracy and physical realism:

1. **Increase grid resolution** to N=128-256 for production runs
2. **Increase `n_pw`** to 5-7 in Phase 5 for quantitatively accurate photonic band gaps
3. **Add domain rotation** in Phase 6 (apply random SO(3) rotation to each domain seed)
4. **Couple Phase 3 output** back to Phase 5 to automatically select the correct Brillouin zone path per symmetry
5. **Export to MEEP or MPB** for full 3-D FDTD or PWE band structure calculations